# 03_multiple_aspects — Multiple aspects and execution order

Three pipeline steps: two `@regular_aspect` and one `@summary_aspect`. Data is passed between them via `state`.

**What's new** (in addition to examples 01–02):

| Concept | Description |
|---------|-------------|
| `@regular_aspect("description")` | Intermediate pipeline step, returns `dict` |
| `state` | Accumulator dict, passed from step to step |
| Execution order | Strictly top to bottom, in declaration order |
| `@result_string("key", required=True)` | Contract: a non-empty string must appear in `state` |

**How `state` works:**
- Pipeline starts with empty `state = {}`
- Each `@regular_aspect` returns a `dict` → that dict **completely replaces** `state`
- Keys from the previous step are **not** copied automatically — pass them explicitly
- `@summary_aspect` reads `state`, but returns a typed `Result`, not a `dict`

**Self-study:** swap `validate_aspect` and `enrich_aspect`. Run again. `enrich_aspect` will try to read `state["cleaned"]`, which doesn't exist yet → `KeyError`. This proves: declaration order = execution order.

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_string
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, ConsoleLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Params & Result

In [ ]:
class ProcessingDomain(BaseDomain):
    name = "processing"
    description = "Input data processing domain"


class ProcessParams(BaseParams):
    raw_input: str = Field(description="Raw input string")


class ProcessResult(BaseResult):
    cleaned: str = Field(description="Cleaned string (stripped and lowercased)")
    enriched: str = Field(description="Enriched version of the cleaned string")
    final: str = Field(description="Final human-readable result description")

## Action — three-step pipeline

**`@regular_aspect`** rules:
- Method name must end with `_aspect`
- Must be `async def`
- Returns a `dict` that **completely replaces** `state`
- Description cannot be empty

**`@result_string("key", required=True)`** — a contract: after this aspect, `state["key"]` must contain a non-empty string. If missing — the machine catches it immediately, before the next step runs. Checkers stack: declare multiple for multiple fields.

Aspects run strictly in the order they are declared in the class — this is a machine guarantee, not a convention.

In [ ]:
@meta(description="Process input string through multiple steps", domain=ProcessingDomain)
@check_roles(GuestRole)
class ProcessInputAction(BaseAction[ProcessParams, ProcessResult]):

    @regular_aspect("Step 1: Strip whitespace and lowercase")
    @result_string("cleaned", required=True)
    async def validate_aspect(self, params, state, box, connections):
        cleaned = params.raw_input.strip().lower()
        await box.info(
            Channel.business,
            "[Step 1] cleaned={%var.cleaned|green}",
            cleaned=cleaned,
        )
        return {"cleaned": cleaned}

    @regular_aspect("Step 2: Enrich data")
    @result_string("cleaned", required=True)
    @result_string("enriched", required=True)
    async def enrich_aspect(self, params, state, box, connections):
        enriched = f"enriched::{state['cleaned']}"
        await box.info(
            Channel.business,
            "[Step 2] enriched={%var.enriched|yellow}",
            enriched=enriched,
        )
        return {
            "cleaned": state["cleaned"],   # explicitly forwarded from Step 1
            "enriched": enriched,
        }

    @summary_aspect("Step 3: Assemble final result")
    async def assemble_summary(self, params, state, box, connections):
        final = f"{state['cleaned']} → {state['enriched']}"
        await box.info(
            Channel.business,
            "[Step 3] final={%var.final|cyan}",
            final=final,
        )
        return ProcessResult(
            cleaned=state["cleaned"],
            enriched=state["enriched"],
            final=final,
        )

## Run

> In Colab, `await` works directly in cells — no need for `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine(
        log_coordinator=LogCoordinator(loggers=[ConsoleLogger()]),
    )
    result = await machine.run(
        Context(),
        ProcessInputAction(),
        ProcessParams(raw_input="  Hello World  "),
    )
    print("\nResult:")
    print(f"  cleaned  = {result.cleaned!r}")
    print(f"  enriched = {result.enriched!r}")
    print(f"  final    = {result.final!r}")

await main()